# Financial Fraud Detection System

## 02 — Data Preparation

This notebook prepares the transaction data used for supervised fine-tuning
and later model evaluation.

The pipeline performs the following operations:

1. connect to the Cifer fraud dataset
2. create a reproducible working subset
3. validate and normalize transaction records
4. create independent training and test partitions
5. balance fraud and legitimate examples independently
6. convert training transactions into Qwen-compatible conversations

The training and test datasets remain independent to prevent evaluation
leakage.

In [1]:
from pathlib import Path
import sys

from datasets import Dataset

current_path = Path.cwd().resolve()

candidate_paths = [
    current_path,
    *current_path.parents,
]

PROJECT_ROOT = next(
    (
        path
        for path in candidate_paths
        if (path / "ml").is_dir()
        and (path / "evaluation").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Unable to locate the Financial Fraud Detection System project root."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.src.data.conversation_format import (
    convert_dataset_to_conversations,
)
from ml.src.data.load_data import load_working_dataset
from ml.src.data.preprocess import (
    preprocess_dataset,
    validate_dataset_schema,
)
from ml.src.data.split_data import (
    get_class_counts,
    prepare_balanced_splits,
)
from ml.src.utils.constants import (
    DATASET_ID,
    RANDOM_SEED,
    TEST_SAMPLES_PER_CLASS,
    TRAIN_SAMPLES_PER_CLASS,
    WORKING_SUBSET_SIZE,
)
from ml.src.utils.seed import set_global_seed

set_global_seed()

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_ID)
print("Random seed:", RANDOM_SEED)

Project root: /Users/ebbejulankapalli/Financial Fraud Detection System
Dataset: CiferAI/Cifer-Fraud-Detection-Dataset-AF
Random seed: 42


## 1. Load the Working Dataset

The original source contains millions of synthetic financial transactions.

For development and fine-tuning, the project selects a reproducible working
subset rather than loading the entire source dataset into memory.

In [2]:
working_dataset = load_working_dataset(
    subset_size=WORKING_SUBSET_SIZE,
    seed=RANDOM_SEED,
)

print(f"Working transactions: {len(working_dataset):,}")
print("Columns:")
print(working_dataset.column_names)

Cifer-Fraud-Detection-Dataset-AF-part-1-(…): reconstructing file:   0%|          |  0.00B /  138MB            

Cifer-Fraud-Detection-Dataset-AF-part-1-(…): downloading bytes:           |  0.00B            

Cifer-Fraud-Detection-Dataset-AF-part-10(…): reconstructing file:   0%|          |  0.00B /  127MB            

Cifer-Fraud-Detection-Dataset-AF-part-10(…): downloading bytes:           |  0.00B            

Cifer-Fraud-Detection-Dataset-AF-part-11(…): reconstructing file:   0%|          |  0.00B /  127MB            

...

Cifer-Fraud-Detection-Dataset-AF-part-8-(…): downloading bytes:           |  0.00B            

Cifer-Fraud-Detection-Dataset-AF-part-9-(…): reconstructing file:   0%|          |  0.00B /  127MB            

Cifer-Fraud-Detection-Dataset-AF-part-9-(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/21000000 [00:00<?, ? examples/s]

Working transactions: 1,000,000
Columns:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


## 2. Validate and Preprocess Transactions

Before creating model datasets, the source schema is validated.

The preprocessing pipeline then:

- removes invalid transactions
- normalizes transaction types
- normalizes numeric features
- validates fraud labels
- retains only the features required by the fraud classifier

Identifiers and `isFlaggedFraud` are intentionally excluded from model
features. In particular, `isFlaggedFraud` could act as a target proxy and
cause leakage.

In [3]:
validate_dataset_schema(working_dataset)

clean_dataset = preprocess_dataset(working_dataset)

print(f"Rows before preprocessing: {len(working_dataset):,}")
print(f"Rows after preprocessing:  {len(clean_dataset):,}")
print()
print("Model columns:")
print(clean_dataset.column_names)
print()
print("Class distribution:")
print(get_class_counts(clean_dataset))

Filtering invalid transactions:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Normalizing transactions:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Rows before preprocessing: 1,000,000
Rows after preprocessing:  1,000,000

Model columns:
['type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud']

Class distribution:
{0: 998700, 1: 1300}


## 3. Create Independent Train and Test Sets

The dataset is split **before balancing**.

This ordering is important.

If transactions were balanced first and then divided carelessly, information
from the sampling procedure could leak between training and evaluation.

The project therefore follows:

**Source → Independent Split → Balance Train → Balance Test**

Each partition is balanced independently.

In [4]:
prepared_splits = prepare_balanced_splits(
    clean_dataset,
    train_samples_per_class=TRAIN_SAMPLES_PER_CLASS,
    test_samples_per_class=TEST_SAMPLES_PER_CLASS,
    seed=RANDOM_SEED,
)

train_dataset = prepared_splits["train"]
test_dataset = prepared_splits["test"]

print("=" * 70)
print("BALANCED DATASET SUMMARY")
print("=" * 70)

print(f"Training transactions: {len(train_dataset):,}")
print("Training classes:", get_class_counts(train_dataset))

print()

print(f"Testing transactions:  {len(test_dataset):,}")
print("Testing classes:", get_class_counts(test_dataset))

print("=" * 70)

Selecting class 0:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Selecting class 1:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Selecting class 0:   0%|          | 0/800000 [00:00<?, ? examples/s]

Selecting class 1:   0%|          | 0/800000 [00:00<?, ? examples/s]

Selecting class 0:   0%|          | 0/200000 [00:00<?, ? examples/s]

Selecting class 1:   0%|          | 0/200000 [00:00<?, ? examples/s]

BALANCED DATASET SUMMARY
Training transactions: 2,080
Training classes: {0: 1040, 1: 1040}

Testing transactions:  520
Testing classes: {0: 260, 1: 260}


## 4. Verify Dataset Separation

Training examples must not appear in the evaluation set.

The reusable splitting pipeline performs the separation before independent
class balancing. This notebook also records the resulting dataset sizes and
class distributions for reproducibility.

In [5]:
train_counts = get_class_counts(train_dataset)
test_counts = get_class_counts(test_dataset)

assert train_counts[0] == train_counts[1], (
    "Training dataset is not balanced."
)

assert test_counts[0] == test_counts[1], (
    "Test dataset is not balanced."
)

print("Training dataset balanced: PASS")
print("Test dataset balanced:     PASS")
print("Independent splitting was performed before balancing: PASS")

Training dataset balanced: PASS
Test dataset balanced:     PASS
Independent splitting was performed before balancing: PASS


## 5. Convert Training Data to Conversational Format

Qwen 2.5-1.5B-Instruct is an instruction-tuned language model.

Instead of training directly on table rows, every transaction is transformed
into a conversation consisting of:

- **User:** transaction details
- **Assistant:** `HIGH` or `LOW`

`HIGH` represents fraud and `LOW` represents a legitimate transaction.

The transaction prompt contains:

- transaction type
- amount
- sender balance before
- sender balance after
- recipient balance before
- recipient balance after

In [6]:
train_conversations = convert_dataset_to_conversations(
    train_dataset,
)

print(
    f"Training conversations created: "
    f"{len(train_conversations):,}"
)

print()
print("Conversation columns:")
print(train_conversations.column_names)

Creating training conversations:   0%|          | 0/2080 [00:00<?, ? examples/s]

Training conversations created: 2,080

Conversation columns:
['type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'messages']


## 6. Inspect a Training Conversation

A single converted example demonstrates exactly what the fine-tuned model
will see during supervised training.

In [7]:
example = train_conversations[0]

print("USER MESSAGE")
print("-" * 70)
print(example["messages"][0]["content"])

print()
print("EXPECTED ASSISTANT RESPONSE")
print("-" * 70)
print(example["messages"][1]["content"])

USER MESSAGE
----------------------------------------------------------------------
Analyze this transaction for fraud risk:
- Type: PAYMENT
- Amount: $2,937.93
- Sender Balance Before: $878.00
- Sender Balance After: $48.00
- Recipient Balance Before: $705,489.50
- Recipient Balance After: $12.14

EXPECTED ASSISTANT RESPONSE
----------------------------------------------------------------------
LOW


## 7. Validate Conversation Labels

The conversational training dataset should contain only the two supported
classification labels:

- `HIGH`
- `LOW`

In [8]:
conversation_labels = [
    row["messages"][1]["content"]
    for row in train_conversations
]

label_counts = {
    "HIGH": conversation_labels.count("HIGH"),
    "LOW": conversation_labels.count("LOW"),
}

print("Conversation label distribution:")
print(label_counts)

assert set(conversation_labels) == {"HIGH", "LOW"}
assert label_counts["HIGH"] == label_counts["LOW"]

print()
print("Conversation label validation: PASS")

Conversation label distribution:
{'HIGH': 1040, 'LOW': 1040}

Conversation label validation: PASS


## 8. Final Data Preparation Summary

At this point the data pipeline has completed:

**Cifer Dataset**

↓

**Working Subset**

↓

**Schema Validation + Preprocessing**

↓

**Independent Train/Test Split**

↓

**Independent Class Balancing**

↓

**Qwen Conversation Conversion**

The resulting training conversations are ready for QLoRA supervised
fine-tuning.

The balanced test dataset remains separate and will later be used to compare
the original Qwen base model against the fine-tuned fraud detector.

### Leakage Controls

The project deliberately excludes:

- transaction identifiers
- source/destination account identifiers
- `isFlaggedFraud`

Only transaction characteristics available to the fraud classifier are used.

The next notebook will configure and run **4-bit QLoRA fine-tuning of
Qwen/Qwen2.5-1.5B-Instruct**.